Created by: Varuna
Date: Oct 2025

# Imports

In [1]:
import pandas as pd
import json
import numpy as np

# Load data

In [2]:
adni = pd.read_csv('/projectnb/vkolagrp/projects/adrd_foundation_model/image_proc/gamlss_oct2025/by_cohort/adni_data_wgamlss.csv')

In [3]:
adni.shape

(1392, 244)

# Data dictionary

In [4]:
with open("adni_dictionary_oct2025_vj.json", 'r') as json_file:
    data_dict = json.load(json_file)

In [5]:
unique_form_ids = set(item['FORM ID'] for item in data_dict.values())
unique_form_ids

{'A1',
 'A3',
 'A5',
 'B1',
 'B2',
 'B4',
 'B5',
 'B6',
 'B7',
 'C',
 'CSF',
 'D1',
 'G1',
 'blood'}

In [6]:
form_dict = {
    'A1': 'Subject Demographics',
    'A3': 'Subject Family History',
    'A5': 'Subject Health History',
    'B1': 'Physical',
    'B2': 'Cardiovascular disease',
    'B4': 'CDR',
    'B5': 'Neuropsychiatric Inventory Questionnaire (NPI-Q)',
    'B6': 'Geriatric Depression Scale (GDS)',
    'B7': 'Functional Assessment Questionnaire (FAQ)',
    'C': 'Neuropsychological Battery Summary Scores',
    'CSF': 'CSF Biomarkers',
    'D1': 'Clinician Diagnosis',
    'G1': 'Genetics',
    'blood': 'Blood-based biomarkers'
}

In [7]:
def collect_skip_cols(d, parent_key=""):
    skip_cols = []
    for key, val in d.items():
        if isinstance(val, dict):
            form_id = val.get("FORM ID")
            val_type = val.get("Type", "").lower() if val.get("Type") else ""
            
            # If this dict has the keys we want, collect
            if form_id in ["B4", "D1", "CSF"] or val_type == "datetime":
                full_key = f"{parent_key}.{key}" if parent_key else key
                skip_cols.append(full_key)
            else:
                # Recurse into nested dicts
                skip_cols.extend(collect_skip_cols(val, parent_key=f"{parent_key}.{key}" if parent_key else key))
    return skip_cols

skip_cols = collect_skip_cols(data_dict)

print(f"Columns to skip ({len(skip_cols)}):")
print(skip_cols)

Columns to skip (14):
['DIAGNOSIS', 'CDMEMORY', 'CDORIENT', 'CDJUDGE', 'CDCOMMUN', 'CDHOME', 'CDCARE', 'CDGLOBAL', 'CDSOB', 'PHC_pTau', 'PHC_AB42', 'AT_class', 'CENTILOIDS', 'AMYLOID_PET_POSITIVE']


In [8]:
skip_cols_cleaned = [k.split('.', 1)[-1] for k in skip_cols]

In [9]:
skip_cols_cleaned

['DIAGNOSIS',
 'CDMEMORY',
 'CDORIENT',
 'CDJUDGE',
 'CDCOMMUN',
 'CDHOME',
 'CDCARE',
 'CDGLOBAL',
 'CDSOB',
 'PHC_pTau',
 'PHC_AB42',
 'AT_class',
 'CENTILOIDS',
 'AMYLOID_PET_POSITIVE']

In [10]:
def create_mapped_json(row, data_dict, form_dict, skip_cols=None, special_fields=None):
    """
    Maps a data row to structured JSON using a nested data dictionary.
    
    Parameters:
    - row: pandas Series or dict containing the data row
    - data_dict: dictionary containing field mappings, can be nested or flat:
                 {field_name: {'FORM ID': str, 'Description': str, 'Values': dict}}
                 OR {section: {field_name: {...}}}
    - form_dict: dictionary mapping form IDs to form descriptions
    - skip_cols: list of columns to skip (optional)
    - special_fields: dict of special field processing functions (optional)
    
    Returns:
    - JSON string of mapped data
    """

    result = {}
    flat_data_dict = {}
    
    # Initialize skip_cols if None
    if skip_cols is None:
        skip_cols = []
    
    def flatten_dict(d, parent_key=''):
        """Recursively flatten nested dictionary structure"""
        for key, value in d.items():
            if isinstance(value, dict) and 'FORM ID' not in value:
                # This is a nested section, recurse into it
                flatten_dict(value, parent_key)
            elif isinstance(value, dict) and 'FORM ID' in value:
                # This is a field definition
                flat_data_dict[key] = value
    
    flatten_dict(data_dict)
    
    for column, value in row.items():
        if column in skip_cols:
            continue

        # Skip if nan or None
        if pd.isna(value) or value is None:
            continue
            
        # Skip if column not in flattened data dictionary
        if column not in flat_data_dict:
            continue
        
        # Check if FORM ID exists in the field definition
        if 'FORM ID' not in flat_data_dict[column]:
            continue
            
        form_id = flat_data_dict[column]['FORM ID']
        
        # Check if form_id exists in form_dict
        if form_id not in form_dict:
            print(f"Warning: FORM ID '{form_id}' for column '{column}' not found in form_dict")
            continue
        
        # Get form description and create form section if needed
        form_desc = form_dict[form_id]
        if form_desc not in result:
            result[form_desc] = {}
            
        # Convert value to appropriate type if specified
        if 'Type' in flat_data_dict[column]:
            try:
                data_type = flat_data_dict[column]['Type'].lower()
                if data_type in ['int', 'integer']:
                    value = int(float(value))
                elif data_type in ['float', 'number']:
                    value = float(value)
                elif data_type in ['str', 'string']:
                    value = str(value)
                elif data_type in ['bool', 'boolean']:
                    value = bool(value)
            except (ValueError, TypeError):
                # If conversion fails, skip this value
                continue
            
        # Get field description
        description = flat_data_dict[column]['Description']
        
        # Handle value mapping if Values key exists
        if 'Values' in flat_data_dict[column]:
            values_def = flat_data_dict[column]['Values']
            
            # Handle different formats of Values
            if isinstance(values_def, dict):
                # Check if it's a Range definition
                if 'Range' in values_def:
                    range_vals = values_def['Range']
                    if len(range_vals) == 2:
                        min_val, max_val = range_vals
                        try:
                            num_value = float(value)
                            # Validate value is within range
                            if min_val <= num_value <= max_val:
                                result[form_desc][description] = value
                            else:
                                print(f"Warning: Value {value} for '{column}' outside range [{min_val}, {max_val}]")
                                continue
                        except (ValueError, TypeError):
                            print(f"Warning: Could not convert value {value} to number for range validation")
                            continue
                # Check for freeform text indicator
                elif "<UND>" in values_def or "<UND>: Any text or numbers" in values_def:
                    result[form_desc][description] = value
                # Standard value mapping
                else:
                    # Try multiple formats to match the value
                    str_value = str(value)
                    # Remove decimal if it's a whole number (e.g., "2.0" -> "2")
                    if '.' in str_value:
                        try:
                            float_val = float(str_value)
                            if float_val.is_integer():
                                str_value = str(int(float_val))
                        except ValueError:
                            pass
                    
                    if str_value in values_def:
                        mapped_value = values_def[str_value]
                        if mapped_value and mapped_value != ".":  # Skip empty or unknown values
                            result[form_desc][description] = mapped_value
                    else:
                        print(f"Warning: Value '{value}' for '{column}' not found in value mapping")
                        continue
            elif isinstance(values_def, list):
                # Values is a list of allowed values or contains "<UND>: Any text or numbers"
                if any("<UND>" in str(item) for item in values_def):
                    result[form_desc][description] = value
                elif value in values_def or float(value) in values_def:
                    result[form_desc][description] = value
                else:
                    print(f"Warning: Value '{value}' for '{column}' not in allowed list")
                    continue
        else:
            # For columns without Values mapping, add value directly
            result[form_desc][description] = value
    
    # Handle special fields if provided
    if special_fields:
        for field_name, processor_func in special_fields.items():
            if field_name in row and pd.notna(row[field_name]):
                special_result = processor_func(row[field_name], row)
                if special_result:
                    if "Imaging evidence" in result:
                        result["Imaging evidence"].update(special_result)
                    else:    
                        result["Imaging evidence"] = special_result
    
    # Remove empty form sections
    result = {k: v for k, v in result.items() if v}
    
    key_order = [form_desc for form_desc in form_dict.values() if form_desc in result]
    ordered_result = {k: result[k] for k in key_order if k in result}

    # Add back any keys not in form_dict (from special_fields)
    for k, v in result.items():
        if k not in ordered_result:
            ordered_result[k] = v

    result = ordered_result
                
    return json.dumps(result)

In [11]:
# Pre-validate and clean data
def validate_and_clean(df, data_dict):
    # First flatten the nested data_dict
    flat_data_dict = {}
    
    def flatten_dict(d):
        """Recursively flatten nested dictionary structure"""
        for key, value in d.items():
            if isinstance(value, dict) and 'FORM ID' not in value:
                # This is a nested section, recurse into it
                flatten_dict(value)
            elif isinstance(value, dict) and 'FORM ID' in value:
                # This is a field definition
                flat_data_dict[key] = value
    
    flatten_dict(data_dict)
    
    # Now validate columns
    for column in df.columns:
        if column in flat_data_dict and 'Values' in flat_data_dict[column]:
            values_def = flat_data_dict[column]['Values']
            
            if isinstance(values_def, dict) and 'Range' in values_def:
                min_val, max_val = values_def['Range']
                mask = (df[column] < min_val) | (df[column] > max_val)
                if mask.any():
                    print(f"Setting {mask.sum()} out-of-range values to NaN in '{column}'")
                    df.loc[mask, column] = np.nan
    
    return df

In [12]:
# Fix DATE column 
def fix_date_column(df):
    """
    DATE should be 0 or 1 (MoCA scoring), but some entries are actual dates.
    Convert date strings to NaN.
    """
    if 'DATE' in df.columns:
        # Identify values that look like dates (contain '-' or '/')
        mask = df['DATE'].astype(str).str.contains('-|/', na=False)
        
        if mask.any():
            print(f"Found {mask.sum()} date strings in DATE column (should be 0/1). Setting to NaN.")
            print(f"Examples: {df.loc[mask, 'DATE'].head().tolist()}")
            df.loc[mask, 'DATE'] = np.nan
    
    return df

In [13]:
# Clean first, then create JSON
adni = validate_and_clean(adni, data_dict)

Setting 2 out-of-range values to NaN in 'TRABERROM'
Setting 2 out-of-range values to NaN in 'AVDEL30MIN'
Setting 6 out-of-range values to NaN in 'PHC_AB42'
Setting 13 out-of-range values to NaN in 'PHC_pTau'


In [14]:
# Apply the fix
adni = fix_date_column(adni)

Found 50 date strings in DATE column (should be 0/1). Setting to NaN.
Examples: ['2021-05-04', '2021-05-06', '2021-05-10', '2021-05-10', '2021-06-02']


In [15]:
# regions = [
# 'FRONTALTransformed', 
# 'HIPPOCAMPUSTransformed', 
# 'INFERIOR_VENTRICLESTransformed', 
# 'MEDIAL_TEMPORALTransformed', 
# 'LATERAL_TEMPORALTransformed', 
# 'MEDIAL_PARIETALTransformed', 
# 'LATERAL_PARIETALTransformed', 
# 'OCCIPITALTransformed']

In [16]:
def process_imaging_summary(value, full_row):
    return {
        "MRI Report": {
            "MRI features extraction procedure": "A Generalized Additive Model for Location, Scale, and Shape (GAMLSS) was fitted to FastSurfer-derived regional brain volumes acquired from 2,087 healthy controls, following the Brain charts for the human lifespan methodology published in Nature (2022). Data were aggregated from five independent cohorts, and the analysis included region-of-interest (ROI) volumes for the medial and lateral temporal lobes, medial and lateral parietal lobes, frontal, occipital lobes, as well as the inferior lateral ventricles and hippocampus. To ensure a robust normative reference, participants with any known neurodegenerative disease were excluded and subjects with extreme brain volumes (>3.5 absolute z-scores) were excluded during quality control prior to model fitting. The Brain Charts pipeline was implemented to model age- and sex-specific normative centile curves for each ROI. Centile scores (scaled 0–1) were computed for both healthy controls and patient samples. Abnormality was defined based on a one-sided z-score threshold: mild atrophy/enlargement for scores corresponding to 1 standard deviation (z=1), moderate for 1.5 SD (z=1.5), and severe for 2 SD (z=2). For all lobar ROIs, lower centiles indicated greater atrophy, while for ventricular ROIs, higher centiles indicated enlargement (i.e., abnormality). The resulting findings are as follows: ",
            "T1 MRI Findings": str(value)
        }
    }

special_fields = {
    'brain_findings_summary': process_imaging_summary
}

In [17]:
adni['patient_summary'] = adni.apply(
    lambda row: create_mapped_json(row, data_dict, form_dict, skip_cols_cleaned, special_fields), 
    axis=1
)

In [18]:
adni['CENTILOIDS'].iloc[5]

32.0

In [19]:
for i, row in adni.iterrows():
    if 'MRI Report' in json.loads(row['patient_summary']):
        print(i)
        break

In [20]:
json.loads(adni.iloc[100]['patient_summary'])

{'Subject Demographics': {'1. Participant Gender': 'Female',
  '2a. Participant Month of Birth': 5,
  '2b. Participant Year of Birth': 1922,
  '5. Participant Education': 12.0,
  '9. Language to be used for testing the Participant': 'English',
  '12. Ethnic Category': 'Not Hispanic or Latino',
  '13. Racial Categories': 'White',
  'Subject age at MRI acquisition': 88.37234770704997},
 'Physical': {'Vitals: 1a. Weight': 136.0,
  'Vitals: 1b. Weight Units': 'pounds',
  'Vitals: Systolic - mmHg': 120.0,
  'Vitals: Diastolic - mmHg': 60.0,
  'Vitals: 4. Seated Pulse Rate (per minute)': 64.0,
  'Vitals: 5. Respirations (per minute)': 18.0,
  'Vitals: 6a. Temperature': 97.0,
  'Vitals: 6b. Temperature Source': 'Oral',
  'Vitals: 6c. Temperature Units': 'Fahrenheit'},
 'Cardiovascular disease': {'Modified Hachinski: 1. Abrupt Onset of Dementia': 'Absent',
  'Modified Hachinski: 2. Stepwise Deterioration of Dementia': 'Absent',
  'Modified Hachinski: 3. Somatic Complaints': 'Absent',
  'Modifi

# Diag summary

In [21]:
def create_diag_summary(row, flat_data_dict):
    """
    Creates a diagnostic summary including diagnosis and related clinical variables.
    """
    # REMOVE THIS LINE: flat_data_dict = flatten_dict(data_dict)
    result = {}
    
    # Define the columns to include in diagnostic summary
    diag_cols = ['DIAGNOSIS', 'CDMEMORY', 'CDORIENT', 'CDJUDGE', 'CDCOMMUN', 
                 'CDHOME', 'CDCARE', 'CDGLOBAL', 'CDSOB', 'PHC_pTau', 'PHC_AB42', 
                 'AT_class', 'CENTILOIDS', 'AMYLOID_PET_POSITIVE']
    
    for col in diag_cols:
        # Skip if column not in row or is NaN
        if col not in row or pd.isna(row[col]):
            continue
        
        # Skip if column not in data dictionary
        if col not in flat_data_dict:
            continue
        
        field_info = flat_data_dict[col]
        description = field_info['Description']
        value = row[col]
        
        # Apply value mapping if available
        if 'Values' in field_info:
            values_def = field_info['Values']
            
            if isinstance(values_def, dict):
                # Handle Range definitions
                if 'Range' in values_def:
                    result[description] = value
                # Handle freeform text
                elif "<UND>" in values_def or "<UND>: Any text or numbers" in values_def:
                    result[description] = value
                # Handle standard value mapping
                else:
                    # Convert to string for mapping
                    str_value = str(value)
                    if '.' in str_value:
                        try:
                            float_val = float(str_value)
                            if float_val.is_integer():
                                str_value = str(int(float_val))
                        except ValueError:
                            pass
                    
                    if str_value in values_def:
                        mapped_value = values_def[str_value]
                        if mapped_value and mapped_value != ".":
                            result[description] = mapped_value
                    else:
                        # If no mapping found, use original value
                        result[description] = value
            else:
                # If Values is not a dict, use original value
                result[description] = value
        else:
            # No Values mapping, use original value
            result[description] = value
    
    return json.dumps(result) if result else json.dumps({})


# Flatten data_dict WITHOUT skip_cols to include everything
flat_data_dict_full = {}

def flatten_dict(d, flat_dict):
    """Recursively flatten nested dictionary structure"""
    for key, value in d.items():
        if isinstance(value, dict) and 'FORM ID' not in value:
            flatten_dict(value, flat_dict)
        elif isinstance(value, dict) and 'FORM ID' in value:
            flat_dict[key] = value

In [22]:
# Flatten data_dict WITHOUT skip_cols to include everything
flat_data_dict_full = {}

def flatten_dict(d, flat_dict):
    """Recursively flatten nested dictionary structure"""
    for key, value in d.items():
        if isinstance(value, dict) and 'FORM ID' not in value:
            # This is a nested section, recurse into it
            flatten_dict(value, flat_dict)
        elif isinstance(value, dict) and 'FORM ID' in value:
            # This is a field definition
            flat_dict[key] = value

In [23]:

flatten_dict(data_dict, flat_data_dict_full)

# Now use flat_data_dict_full for diag_summary
adni['diag_summary'] = adni.apply(
    lambda row: create_diag_summary(row, flat_data_dict_full), 
    axis=1
)


In [24]:
print(adni['diag_summary'].iloc[66])

{"Which best describes the participant's current diagnosis?": "Cognitively normal", "CDR: Score Memory": 0.0, "CDR: Score Orientation": 0.0, "CDR: Score Judgment and Problem Solving": 0.0, "CDR: Score Community Affairs": 0.0, "CDR: Score Home and Hobbies": 0.0, "CDR: Score Personal Care": 0.0, "Global CDR": 0.0, "CDR: Sum of Boxes": 0.0, "Summary cortical SUVR normalized by whole cerebellum and transformed to Centiloids": -7.0, "Centiloids value thresholded at 24 CL": "Amyloid PET negative"}


In [25]:
# adni.to_csv('/projectnb/vkolagrp/projects/adrd_foundation_model/json_testing_data/adni_oct_2025.csv', index=False)

In [26]:
adni.to_csv('/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/adni/test_gamlss.csv', index=False)